# 01 — Data Forensics & EDA

The brief is explicit: *"your first major task is data forensics."*

This notebook hunts for legacy SFA/ERP artifacts in the raw exports before any modeling is done. Each anomaly class we find here becomes either:
- a **reusable DQ check** (in `src/processing/data_quality.py`), or
- a **feature** the modeling layer uses to detect censoring.

### What we look for

1. Schema integrity — missing/extra columns, type drift
2. Duplicates — exact, near-duplicates, transaction-level vs outlet-day-level
3. Ghost entries — long runs of identical non-zero volume per outlet
4. Distributor-wide blackouts — days where ~all outlets of a distributor record zero
5. Credit-cap fingerprints — clean multiples of 100/500/1000 dominating an outlet's volume
6. Outlet master decay — outlets in transactions but not in master, multiple coords per outlet
7. Coordinate validity — Sri Lanka bounding-box checks
8. Stockout signatures — drops correlated with distributor-level SKU gaps
9. Round-number / default-value spikes
10. Date sanity — future dates, year-2000 defaults, weekend-only patterns

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.utils.io import read_parquet, read_csv_resilient, normalize_columns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Schema & basic profile

Bronze is just raw-as-parquet, so this is a fast first look at what we're working with.

In [ ]:
datasets = {}
for name in config.SOURCE_FILES:
    p = config.BRONZE_DIR / f"{name}.parquet"
    if p.exists():
        datasets[name] = read_parquet(p)
        print(f"{name:15s} rows={len(datasets[name]):>8,}  cols={datasets[name].shape[1]}")
        print("   columns:", list(datasets[name].columns))
    else:
        print(f"{name:15s} MISSING ({p})")

In [ ]:
tx = datasets.get("transactions", pd.DataFrame())
if not tx.empty:
    tx["date"] = pd.to_datetime(tx["date"], errors="coerce")
    print("date range:", tx["date"].min(), "->", tx["date"].max())
    print("\nnulls per column:")
    print(tx.isna().mean().sort_values(ascending=False).head(15))
    display(tx.head())

## 2. Ghost-entry hunt — constant runs

If the SFA app pre-fills the same volume across days, an outlet will have long runs of identical non-zero values. We use the registered `constant_run` DQ check to find these.

In [ ]:
from src.processing.data_quality import check_constant_runs

if not tx.empty:
    passing, rejected = check_constant_runs(
        tx, group_keys=["outlet_id"], order_col="date",
        value_col="volume_liters", min_run=7,
    )
    print(f"constant-run rejections: {len(rejected):,} / {len(tx):,} ({len(rejected)/max(len(tx),1):.2%})")
    if not rejected.empty:
        per_outlet = rejected.groupby("outlet_id").size().sort_values(ascending=False)
        print("top affected outlets:")
        print(per_outlet.head(10))

## 3. Distributor blackout days

If 95%+ of a distributor's outlets log zero on a given day, that day is almost certainly a connectivity / sync issue rather than real demand collapse.

In [ ]:
from src.processing.data_quality import check_distributor_blackout

if not tx.empty and "distributor_id" in tx.columns:
    passing, rejected = check_distributor_blackout(
        tx,
        distributor_col="distributor_id", date_col="date",
        outlet_col="outlet_id", value_col="volume_liters",
        blackout_threshold=0.95,
    )
    print(f"blackout rejections: {len(rejected):,}")
    if not rejected.empty:
        bd = rejected.groupby(["distributor_id", "date"]).size().reset_index(name="rows")
        print("blackout days per distributor:")
        print(bd.groupby("distributor_id")["date"].nunique().sort_values(ascending=False))

## 4. Credit-cap fingerprints

Outlets whose volume is dominated by clean multiples of 100/500/1000 are almost certainly being capped by credit/route logic, not true demand. These months get tagged as `is_constrained` downstream so they can't be mistaken for ceilings.

In [ ]:
from src.processing.data_quality import check_credit_cap_signature

if not tx.empty:
    tagged, _ = check_credit_cap_signature(
        tx, group_keys=["outlet_id"], value_col="volume_liters",
        modulos=config.DQ_CONFIG["round_number_suspicion_modulos"],
    )
    rate = tagged["_credit_cap_flag"].mean() if "_credit_cap_flag" in tagged.columns else 0.0
    print(f"credit-cap-tagged rows: {rate:.2%} of transactions")
    if "_credit_cap_flag" in tagged.columns:
        flagged_outlets = tagged[tagged["_credit_cap_flag"]]["outlet_id"].nunique()
        print(f"distinct outlets with credit-cap signature: {flagged_outlets:,}")

## 5. Outlet master sanity

- Outlets in transactions but missing from master ⇒ referential integrity failure.
- Outlets outside the Sri Lanka bounding box ⇒ coordinate corruption.
- Outlets with duplicate `outlet_id` rows ⇒ master-data decay.

In [ ]:
outlets = datasets.get("outlets", pd.DataFrame())
if not outlets.empty:
    print(f"outlets rows: {len(outlets):,}")
    print(f"distinct outlet_id: {outlets['outlet_id'].nunique():,}")
    print(f"distinct distributor_id: {outlets['distributor_id'].nunique() if 'distributor_id' in outlets.columns else 'n/a'}")
    if "province" in outlets.columns:
        print(outlets["province"].value_counts(dropna=False))
    if {"latitude", "longitude"}.issubset(outlets.columns):
        lat_bounds = config.SCHEMAS["outlets"].numeric_ranges["latitude"]
        lon_bounds = config.SCHEMAS["outlets"].numeric_ranges["longitude"]
        oob = (
            ~outlets["latitude"].between(*lat_bounds)
            | ~outlets["longitude"].between(*lon_bounds)
        )
        print(f"out-of-bounds coords: {int(oob.sum()):,}")

## 6. Run the full Silver pipeline & review the rejection summary

This is the **single audit trail** the judges will look at — every check, every row count, every reason.

In [ ]:
from src.processing.silver_cleaning import run_silver_cleaning
import json

summary = run_silver_cleaning()
rows = []
for ds in summary["datasets"]:
    if ds.get("status") != "OK":
        continue
    for step in ds["summary"]:
        rows.append(step)
summary_df = pd.DataFrame(rows)
display(summary_df)

In [ ]:
rejected_files = sorted(config.REJECTED_DIR.glob("*.parquet"))
print(f"rejected parquet files: {len(rejected_files)}")
for f in rejected_files:
    df = read_parquet(f)
    print(f"  {f.name}: {len(df):,} rows | reasons: {df['failure_reason'].value_counts().head(3).to_dict()}")